# FieldMedic Gemma 4 E2B fine-tune

Run on Colab with **T4 / L4 / A100 GPU runtime**.

Steps:
1. Set runtime: `Runtime → Change runtime type → T4 GPU` (free) or A100 (faster, paid)
2. Run all cells top-to-bottom
3. When prompted, upload `dataset_expanded.jsonl` (use the Files panel on the left)
4. Wait for training (~30-90 min on T4 with 420 examples)
5. Last cell saves the LoRA adapter to your Google Drive

In [ ]:
# 1. Verify GPU
!nvidia-smi

In [ ]:
# 2. Install Unsloth + deps. Takes ~3-5 min on Colab.
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth trl peft accelerate bitsandbytes datasets

In [ ]:
# 3. Mount Google Drive so we can save outputs.
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_BASE = '/content/drive/MyDrive/FieldMedic'
os.makedirs(OUTPUT_BASE, exist_ok=True)
print('Outputs will be saved to:', OUTPUT_BASE)

In [ ]:
# 4. Upload the dataset.
# Click the Files icon on the left sidebar, then drag dataset_expanded.jsonl into /content/.
# Or run this cell to use the upload picker:
from google.colab import files
uploaded = files.upload()
print('Files in /content:', os.listdir('/content'))

In [ ]:
# 5. Confirm dataset loaded correctly.
from datasets import load_dataset
ds = load_dataset('json', data_files='/content/dataset_expanded.jsonl', split='train')
print(f'Loaded {len(ds)} examples')
print('Sample:', ds[0])

In [ ]:
# 6. Load Gemma 4 E2B with 4-bit quantization.
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='google/gemma-4-E2B-it',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print('Base model loaded.')

In [ ]:
# 7. Wrap with LoRA.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

In [ ]:
# 8. Format dataset using the model's chat template.
def fmt(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

ds = ds.map(fmt)
print('Formatted. First example text:', ds[0]['text'][:300])

In [ ]:
# 9. Train. ~30-90 min on T4 for 420 examples × 3 epochs.
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir=f'{OUTPUT_BASE}/train_output',
        report_to='none',
    ),
)
trainer.train()

In [ ]:
# 10. Save the LoRA adapter to Drive (~50 MB).
ADAPTER_OUT = f'{OUTPUT_BASE}/fieldmedic-gemma-4-e2b-lora'
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f'LoRA adapter saved to: {ADAPTER_OUT}')
!ls -lh '{ADAPTER_OUT}'

In [ ]:
# 11. Quick eval — run a few sample prompts and inspect output.
FastLanguageModel.for_inference(model)

samples = [
    'tiny paper cut on my finger',
    'he is not breathing',
    'what dose of paracetamol for 30kg child',
    'my dog has a wound on his paw',
]
for s in samples:
    prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': s}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n=== {s} ===\n{text[:500]}')

In [ ]:
# 12. Optional — also save the merged 16-bit weights (needed before LiteRT export).
# Skip if Drive is tight on space (merged model is ~10 GB).
MERGED_OUT = f'{OUTPUT_BASE}/fieldmedic-gemma-4-e2b-merged'
model.save_pretrained_merged(MERGED_OUT, tokenizer, save_method='merged_16bit')
print(f'Merged HF weights saved to: {MERGED_OUT}')
print('Next: convert this to .litertlm using ai-edge-torch (separate step, see merge_and_export.py).')